# Fisher information and the Cramér–Rao lower bound

## Why curvature of the log-likelihood controls estimation precision

The shape of the log-likelihood near its maximum tells you something fundamental about estimation. A sharply peaked log-likelihood means the data strongly constrain the parameter; a flat one means many parameter values are about equally compatible with what was observed.

In the previous post on likelihood and the score, we introduced the likelihood, the log-likelihood, and the score, and saw that the MLE maximises the likelihood. This post takes up the natural next question: how precisely can we estimate the parameter? Fisher information is the quantity that answers this, and the Cramér–Rao lower bound is the statement it leads to.

All figures are saved into `content/blog/fisher-crlb/` next to `index.md` (Hugo page bundle). After regenerating PNGs, run **`hugo`** from the repository root.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # headless backend

import matplotlib.pyplot as plt
import numpy as np

for _style in ("seaborn-v0_8-whitegrid", "seaborn-whitegrid", "ggplot"):
    try:
        plt.style.use(_style)
        break
    except OSError:
        continue
plt.rcParams["figure.dpi"] = 140
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

RNG = np.random.default_rng(20260407)


def locate_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "content").exists() and (cwd / "notebooks").exists():
        return cwd
    if cwd.name == "notebooks" and (cwd.parent / "content").exists():
        return cwd.parent
    for parent in [cwd, *cwd.parents]:
        if (parent / "content").exists() and (parent / "notebooks").exists():
            return parent
        if (parent / "hugo.toml").exists() and (parent / "content").exists():
            return parent
    raise RuntimeError("Could not locate repository root.")


ROOT = locate_root()
OUTDIR = ROOT / "content" / "blog" / "fisher-crlb"
OUTDIR.mkdir(parents=True, exist_ok=True)


def savefig(name):
    path = (OUTDIR / name).resolve()
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight", dpi=180, facecolor="white")
    plt.close()
    print(path.relative_to(ROOT))
    return path


ROOT, OUTDIR

## Fisher information

### Scalar case

Let $X \sim f(\cdot, \theta)$ with scalar parameter $\theta \in \Theta \subseteq \mathbb{R}$. Recall that the score for a single observation is

$$
S(\theta) = \frac{\partial}{\partial\theta} \log f(X, \theta),
$$

and that the expected score is zero: $\mathbb{E}_\theta[S(\theta)] = 0$. The **Fisher information** is the variance of the score:

$$
I(\theta) = \mathbb{E}_\theta\!\left[S(\theta)^2\right] = \text{Var}_\theta(S(\theta)).
$$

Under regularity conditions that allow differentiation under the integral, there is an equivalent and often more convenient form:

$$
I(\theta) = -\mathbb{E}_\theta\!\left[\frac{\partial^2}{\partial\theta^2} \log f(X, \theta)\right].
$$

This is the expected negative curvature of the log-likelihood. Both expressions give the same number, but they offer different intuitions. The first says Fisher information is the spread of the score; the second says it is the average sharpness of the log-likelihood.

This definition is for **one observation**. For i.i.d. data $X_1, \dots, X_n$, Fisher information scales linearly:

$$
I_n(\theta) = n\, I_1(\theta).
$$

### Multivariate case

When $\theta \in \mathbb{R}^p$, the score becomes a vector $S(\theta) = \nabla_\theta \log f(X, \theta)$, and Fisher information becomes a $p \times p$ matrix:

$$
I(\theta) = \mathbb{E}_\theta\!\left[S(\theta)\, S(\theta)^\top\right].
$$

Under regularity conditions:

$$
I(\theta) = -\mathbb{E}_\theta\!\left[\nabla^2_\theta \log f(X, \theta)\right].
$$

The matrix version is not just a notational upgrade. In higher dimensions, parameter uncertainty has direction: the data may be very informative about one combination of parameters and much less informative about another. The Fisher information matrix encodes this directional structure. For i.i.d. data, additivity still holds: $I_n(\theta) = n\, I_1(\theta)$.

> The equivalence of the two forms of Fisher information is proved in Appendix A1 below, and the additivity under i.i.d. sampling in Appendix A2, both following Rajen Shah's notes.

## What Fisher information is measuring

Fisher information measures the sensitivity of the statistical model to small changes in the parameter.

If changing $\theta$ slightly causes the distribution $f(\cdot, \theta)$ to change a lot, then data drawn from the model should be informative about $\theta$. If changing $\theta$ barely changes the distribution, the data cannot tell you much about where $\theta$ is.

This sensitivity has two equivalent readings:

- **Variance of the score.** If the score $S(\theta)$ varies a lot across different samples, the model is responding strongly to the data at that parameter value.
- **Expected negative curvature.** If the log-likelihood is sharply curved on average, nearby parameter values are easy to distinguish from $\theta$ itself.

The second interpretation is especially geometric, and motivates the next section.

> Fisher information measures how sharply the statistical model reacts, on average, to small changes in the parameter.

## Why curvature matters

The log-likelihood is a random curve, and the MLE is its maximiser. The shape of that curve near its maximum should affect how much the estimator varies from sample to sample.

If the log-likelihood is sharply peaked, the MLE is tightly constrained: moving $\theta$ slightly away from the maximum causes a steep drop in log-likelihood, so different samples tend to produce similar maximisers. If the log-likelihood is flat, many parameter values achieve nearly the same likelihood, and the MLE wanders more across repeated samples.

> A sharper log-likelihood means more local information about the parameter, and lower estimator variance.

To make this concrete, consider the Gaussian location model with known variance:

$$
X_1, \dots, X_n \sim N(\theta, \sigma^2),
$$

with true parameter $\theta_0 = 0$. The log-likelihood is

$$
\ell_n(\theta) = -\frac{1}{2\sigma^2} \sum_{i=1}^n (X_i - \theta)^2 + C,
$$

so its curvature is constant:

$$
\ell_n''(\theta) = -\frac{n}{\sigma^2}.
$$

Smaller $\sigma^2$ means a more negative second derivative, i.e. sharper curvature. The MLE is $\hat\theta = \bar{X}$, with variance $\text{Var}(\hat\theta) = \sigma^2 / n$. Everything lines up: sharper curvature corresponds directly to smaller estimator variance.

We illustrate sample log-likelihoods by drawing $m = 10$ independent samples of size $n = 20$ from $N(0,\sigma_1^2)$ and from $N(0,\sigma_2^2)$ with $\sigma_1^2 = 1$ and $\sigma_2^2 = 10$. Variance is treated as known; only the mean $\theta$ is unknown in the model. The code plots the $\theta$-dependent part of $\ell_n(\theta)$ (same kernel as above, omitting any additive constant). The left panel uses $\sigma_1^2$; the right uses $\sigma_2^2$. Red dots mark $\hat\theta = \bar{X}$ at the peak of each curve. **$n = 20$**, **$m = 10$**, fixed RNG seed for reproducibility.

In [ ]:
# Figure 1: sample log-likelihoods – Gaussian location, known variance
theta0, n_obs, n_rep = 0.0, 20, 10
tgrid = np.linspace(-2.5, 2.5, 500)

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.4))
for ax, s2, lab in zip(
    axes,
    [1.0, 10.0],
    [r"known variance $\sigma^2 = 1$", r"known variance $\sigma^2 = 10$"],
):
    for _ in range(n_rep):
        x = RNG.normal(theta0, np.sqrt(s2), size=n_obs)
        ll = -0.5 / s2 * np.sum((x[:, None] - tgrid[None, :]) ** 2, axis=0)
        mle = x.mean()
        ll_mle = -0.5 / s2 * np.sum((x - mle) ** 2)
        ax.plot(tgrid, ll, color="#2563eb", alpha=0.45, lw=1.2)
        ax.scatter([mle], [ll_mle], color="#dc2626", s=25, zorder=3, alpha=0.8)
    ax.axvline(theta0, color="#111827", ls="--", lw=1.2, alpha=0.5,
               label=rf"true $\theta_0 = {theta0:.0f}$")
    ax.set_xlabel(r"$\theta$")
    ax.set_title(lab, fontsize=11)
    ax.legend(frameon=False, fontsize=9)
axes[0].set_ylabel(r"sample log-likelihood $\ell_n(\theta)$")
savefig("curvature-comparison.png")

This figure is an intuition builder, not a proof. The curves shown are realised log-likelihoods from particular samples, while Fisher information is an expected quantity. But the picture makes a concrete point: curvature and estimator variability are linked.

## From observed curvature to Fisher information

The figure shows realised log-likelihood curves from specific samples. Fisher information is not the curvature of any single realised curve. It is the **expected** negative curvature:

$$
I(\theta) = -\mathbb{E}_\theta\!\left[\frac{\partial^2}{\partial\theta^2} \log f(X, \theta)\right].
$$

Fisher information is the average local sharpness that the model tends to produce at parameter value $\theta$. The distinction matters:

- **Observed curvature** depends on the particular sample. It is different for each dataset.
- **Fisher information** is a property of the model. It averages curvature over all possible data the model could generate.

In practice, one often also uses the **observed information** $J(\hat\theta) = -\ell_n''(\hat\theta)$, the curvature of the realised log-likelihood evaluated at the MLE. But the theoretical object that appears in the Cramér–Rao bound is Fisher information.

> The figure explains why curvature should matter. Fisher information formalises this by averaging curvature over repeated samples from the model.

## Why information adds for i.i.d. samples

For i.i.d. observations, the log-likelihood is a sum of per-observation contributions, and the score adds accordingly:

$$
S_n(\theta) = \sum_{i=1}^n S_i(\theta),
$$

where $S_i(\theta) = \frac{\partial}{\partial\theta} \log f(X_i, \theta)$. Since the $S_i$ are independent and mean-zero, the variance of the sum is the sum of the variances:

$$
I_n(\theta) = \text{Var}_\theta(S_n(\theta)) = \sum_{i=1}^n \text{Var}_\theta(S_i(\theta)) = n\, I_1(\theta).
$$

Independent observations contribute independent pieces of evidence about the parameter, so information accumulates linearly.

> If one observation gives you a certain amount of local information about the parameter, then $n$ independent observations give you $n$ times as much.

This is one reason why variances of efficient estimators typically scale like $1/n$: information grows linearly with sample size, so uncertainty shrinks at the same rate.

## The Cramér–Rao lower bound

**Theorem (Cramér–Rao lower bound).** *Suppose the model $\{P_\theta : \theta \in \Theta\}$ satisfies regularity conditions allowing interchange of differentiation and integration. If $\hat\theta$ is an unbiased estimator of a scalar parameter $\theta$, then*

$$
\text{Var}_\theta(\hat\theta) \geq \frac{1}{I_n(\theta)}.
$$

*In the i.i.d. case with $n$ observations, this becomes*

$$
\text{Var}_\theta(\hat\theta) \geq \frac{1}{n\, I_1(\theta)}.
$$

This is a lower bound on the variance of **any** unbiased estimator. No matter how cleverly an estimator is constructed, if it is unbiased then its variance cannot go below the reciprocal of the Fisher information. The proof uses the Cauchy–Schwarz inequality applied to the covariance between the estimator and the score (see Appendix A3 below).

> The Cramér–Rao bound turns Fisher information into a hard limit: if the model contains only a certain amount of information about the parameter, no unbiased estimator can beat that limit.

### More general forms

The bound extends to estimating a function of the parameter. If $T$ is an unbiased estimator of $g(\theta)$, then

$$
\text{Var}_\theta(T) \geq \frac{(g'(\theta))^2}{I_n(\theta)}.
$$

In higher dimensions with $\theta \in \mathbb{R}^p$, the bound takes a matrix form. For any unbiased estimator $\hat\theta$ of $\theta$,

$$
\text{Cov}_\theta(\hat\theta) \succeq I_n(\theta)^{-1},
$$

meaning the difference $\text{Cov}_\theta(\hat\theta) - I_n(\theta)^{-1}$ is positive semi-definite. The inverse Fisher information matrix plays the role of the canonical covariance lower bound.

## Intuition behind the bound

The Cramér–Rao bound has a clean statistical interpretation.

If the likelihood is flat around $\theta$, many nearby parameter values produce similar likelihoods. The data cannot effectively distinguish between them, so no unbiased estimator can concentrate tightly around the truth. If the likelihood is sharp, nearby parameter values produce noticeably different likelihoods, and lower variance becomes possible.

Fisher information quantifies this local distinguishability. The Cramér–Rao bound says that distinguishability sets a floor on estimator variance.

The bound does **not** say every estimator attains it. It says no unbiased estimator can go below it. Whether the bound is achievable depends on the model.

> The Cramér–Rao bound is not a recipe for constructing an estimator. It is a statement about what is fundamentally achievable.

## Achieving the bound

If an unbiased estimator achieves the Cramér–Rao lower bound, it must have the smallest possible variance among all unbiased estimators. It is automatically a **uniformly minimum-variance unbiased estimator** ([UMVUE](https://en.wikipedia.org/wiki/Minimum-variance_unbiased_estimator)).

The reasoning is direct: the estimator is unbiased, and the CRLB says no unbiased estimator can have smaller variance.

> Achieving the Cramér–Rao lower bound is a certificate of optimality: among unbiased estimators, there is nowhere left to improve.

It is worth noting that unbiasedness is itself a restriction. Some biased estimators can achieve lower mean squared error by trading a small bias for a larger reduction in variance. But within the unbiased class, the CRLB is the definitive benchmark.

## Worked example: Bernoulli model

Let $X_1, \dots, X_n \stackrel{\text{i.i.d.}}{\sim} \text{Bern}(\theta)$, with $\theta \in (0,1)$.

### Likelihood and MLE

The log-likelihood is

$$
\ell_n(\theta) = \left(\sum_{i=1}^n X_i\right) \log\theta + \left(n - \sum_{i=1}^n X_i\right) \log(1-\theta).
$$

Setting the score equal to zero gives the MLE $\hat\theta = \bar{X}$.

### Fisher information

For a single observation, $\ell(\theta; X) = X\log\theta + (1-X)\log(1-\theta)$. The second derivative is

$$
\frac{\partial^2}{\partial\theta^2} \ell(\theta; X) = -\frac{X}{\theta^2} - \frac{1-X}{(1-\theta)^2}.
$$

Taking the expectation using $\mathbb{E}_\theta(X) = \theta$:

$$
I_1(\theta) = \frac{1}{\theta(1-\theta)}.
$$

For $n$ observations, $I_n(\theta) = n / [\theta(1-\theta)]$.

### The bound

The Cramér–Rao lower bound gives

$$
\text{Var}_\theta(\hat\theta) \geq \frac{1}{I_n(\theta)} = \frac{\theta(1-\theta)}{n}.
$$

But $\text{Var}_\theta(\bar{X}) = \theta(1-\theta)/n$, so the MLE achieves the bound exactly.

The MLE $\bar{X}$ is unbiased, achieves the Cramér–Rao lower bound, and is therefore UMVUE for $\theta$ in the Bernoulli model.

The next figure does not draw log-likelihoods. The left panel plots **Fisher information per observation** $I_1(\theta) = 1/[\theta(1-\theta)]$ as a function of $\theta$, so you can see how much information the model carries near the boundary versus near $1/2$. The right panel plots the **CRLB** $\theta(1-\theta)/n$ for $n = 20$ (blue curve) together with **Monte Carlo estimates** of $\text{Var}(\bar{X})$ at several values of $\theta$ (red dots). Each dot uses many independent Bernoulli samples of size $n = 20$ at that $\theta$. If the theory and the simulation agree, the dots sit on the curve, which is what we expect because the MLE attains the bound.

In [ ]:
# Figure 2: Bernoulli Fisher information and CRLB
tb = np.linspace(0.01, 0.99, 500)
I1 = 1.0 / (tb * (1 - tb))
n_b = 20
crlb = tb * (1 - tb) / n_b

n_sim = 50_000
theta_pts = np.linspace(0.05, 0.95, 19)
emp_vars = []
for th in theta_pts:
    samples = RNG.binomial(1, th, size=(n_sim, n_b))
    emp_vars.append(samples.mean(axis=1).var())

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.4))

axes[0].plot(tb, I1, color="#1d4ed8", lw=2)
axes[0].set_xlabel(r"$\theta$")
axes[0].set_ylabel(r"$I_1(\theta)$")
axes[0].set_title("Fisher information per observation", fontsize=11)
axes[0].set_ylim(0, 30)

axes[1].plot(tb, crlb, color="#1d4ed8", lw=2,
             label=r"CRLB $= \theta(1-\theta)/n$")
axes[1].scatter(theta_pts, emp_vars, color="#dc2626", s=30, zorder=3, alpha=0.8,
                label=r"simulated Var$(\bar X)$")
axes[1].set_xlabel(r"$\theta$")
axes[1].set_ylabel("variance")
axes[1].set_title(f"CRLB vs MLE variance ($n = {n_b}$)", fontsize=11)
axes[1].legend(frameon=False, fontsize=9)

savefig("bernoulli-fisher-info.png")

## A note on higher dimensions

In higher dimensions, uncertainty is directional. The Fisher information matrix $I(\theta) \in \mathbb{R}^{p \times p}$ captures how informative the data are about different combinations of parameters. Its inverse $I(\theta)^{-1}$ gives the covariance lower bound in the multivariate Cramér–Rao inequality.

This also explains the form of the asymptotic distribution of the MLE. Under regularity conditions,

$$
\sqrt{n}\,(\hat\theta - \theta_0) \to_d N\!\left(0,\, I_1(\theta_0)^{-1}\right),
$$

so the asymptotic covariance is exactly the inverse Fisher information per observation. The MLE is asymptotically efficient: it achieves the Cramér–Rao bound in the limit.

## Conclusion

Fisher information can be defined as the variance of the score or as the expected negative curvature of the log-likelihood. Both definitions capture the same object: a measure of how much the data can tell you about the parameter.

Curvature matters because sharper log-likelihoods make nearby parameter values easier to distinguish, and this translates directly into how tightly an estimator can concentrate. Information adds linearly across independent observations, which is why estimator precision improves with sample size.

The Cramér–Rao lower bound converts Fisher information into a hard limit on the variance of unbiased estimators. If an estimator achieves this bound, it is optimal among unbiased estimators. In the Bernoulli model, the sample mean does exactly that.

> Fisher information is the mathematical object that converts the geometry of the likelihood into a precise statement about the limits of estimation.

## Appendix

The proofs below follow the presentation in Rajen Shah, *Principles of Statistics*, Section 1.2.

### A1: Equivalence of the two forms of Fisher information

We show that $I(\theta) = \mathbb{E}_\theta[S(\theta)^2] = -\mathbb{E}_\theta[\ell''(\theta)]$ in the scalar case.

*Proof.* Compute the second derivative of the log-density:

$$
\frac{\partial^2}{\partial\theta^2} \log f(x, \theta) = \frac{\frac{\partial^2}{\partial\theta^2} f(x, \theta)}{f(x, \theta)} - \left(\frac{\frac{\partial}{\partial\theta} f(x, \theta)}{f(x, \theta)}\right)^2.
$$

Under regularity conditions, interchanging differentiation and integration gives

$$
\int \frac{\partial^2}{\partial\theta^2} f(x, \theta)\, dx = \frac{\partial^2}{\partial\theta^2} \int f(x, \theta)\, dx = 0,
$$

so the expectation of the first term vanishes. Therefore

$$
-\mathbb{E}_\theta\!\left[\frac{\partial^2}{\partial\theta^2} \log f(X, \theta)\right] = \mathbb{E}_\theta\!\left[\left(\frac{\partial}{\partial\theta} \log f(X, \theta)\right)^2\right] = I(\theta). \quad\square
$$

### A2: Additivity under i.i.d. sampling

*Proof.* For i.i.d. data $X_1, \dots, X_n \sim f(\cdot, \theta)$, the score is $S_n(\theta) = \sum_{i=1}^n \nabla_\theta \log f(X_i, \theta)$. The Fisher information matrix is

$$
I_n(\theta) = \mathbb{E}_\theta[S_n(\theta)\, S_n(\theta)^\top] = \sum_{i=1}^n \sum_{j=1}^n \mathbb{E}_\theta\!\left[\nabla_\theta \log f(X_i, \theta) \cdot \nabla_\theta \log f(X_j, \theta)^\top\right].
$$

The terms $\nabla_\theta \log f(X_i, \theta)$ are i.i.d. and mean-zero, so the cross terms ($i \neq j$) vanish:

$$
I_n(\theta) = \sum_{i=1}^n \mathbb{E}_\theta\!\left[\nabla_\theta \log f(X_i, \theta) \cdot \nabla_\theta \log f(X_i, \theta)^\top\right] = n\, I_1(\theta). \quad\square
$$

### A3: Proof of the scalar Cramér–Rao lower bound

The key ingredient is the following lemma, which holds under regularity conditions allowing interchange of differentiation and integration.

**Lemma.** *For any function $g$ of the data, $\mathbb{E}_\theta[S(\theta)\, g(X)] = \frac{\partial}{\partial\theta} \mathbb{E}_\theta[g(X)]$.*

*Proof of lemma.* We have $\mathbb{E}_\theta[S(\theta)\, g(X)] = \int \frac{\partial}{\partial\theta} \log f(x, \theta) \cdot g(x) \cdot f(x, \theta)\, dx = \int \frac{\partial}{\partial\theta} f(x, \theta) \cdot g(x)\, dx = \frac{\partial}{\partial\theta} \int f(x, \theta) \cdot g(x)\, dx = \frac{\partial}{\partial\theta} \mathbb{E}_\theta[g(X)]$. $\square$

*Proof of the CRLB.* Let $\hat\varphi$ be any estimator. Apply the lemma with $g(X) = \hat\varphi(X)$:

$$
\text{Cov}_\theta(S(\theta),\, \hat\varphi) = \mathbb{E}_\theta[S(\theta) \cdot \hat\varphi] = \frac{\partial}{\partial\theta} \mathbb{E}_\theta[\hat\varphi],
$$

where the first equality uses $\mathbb{E}_\theta[S(\theta)] = 0$. By the Cauchy–Schwarz inequality,

$$
\left(\frac{\partial}{\partial\theta} \mathbb{E}_\theta[\hat\varphi]\right)^2 \leq \text{Var}_\theta(S(\theta)) \cdot \text{Var}_\theta(\hat\varphi) = I(\theta) \cdot \text{Var}_\theta(\hat\varphi).
$$

Rearranging:

$$
\text{Var}_\theta(\hat\varphi) \geq \frac{\left(\frac{\partial}{\partial\theta} \mathbb{E}_\theta[\hat\varphi]\right)^2}{I(\theta)}.
$$

If $\hat\varphi$ is unbiased for $g(\theta)$, then $\mathbb{E}_\theta[\hat\varphi] = g(\theta)$ and the bound becomes $\text{Var}_\theta(\hat\varphi) \geq (g'(\theta))^2 / I(\theta)$. Taking $g(\theta) = \theta$ gives $\text{Var}_\theta(\hat\theta) \geq 1/I(\theta)$. For $n$ i.i.d. observations, replace $I(\theta)$ with $I_n(\theta) = n\, I_1(\theta)$. $\square$

### A4: Multivariate Cramér–Rao lower bound

*Proof.* Let $\hat\varphi$ estimate $\varphi(\theta) \in \mathbb{R}$ and write $b = \nabla_\theta \mathbb{E}_\theta[\hat\varphi]$. Let $v = I(\theta)^{-1/2} u$ for an arbitrary unit vector $u \in \mathbb{R}^p$. The multivariate version of the lemma gives

$$
v^\top b = v^\top \mathbb{E}_\theta[S(\theta)\, \hat\varphi] = \mathbb{E}_\theta\!\left[v^\top S(\theta) \cdot (\hat\varphi - \mathbb{E}_\theta[\hat\varphi])\right],
$$

where the last step uses $\mathbb{E}_\theta[S(\theta)] = 0$. By Cauchy–Schwarz:

$$
(v^\top b)^2 \leq \mathbb{E}_\theta[(v^\top S(\theta))^2] \cdot \text{Var}_\theta(\hat\varphi) = v^\top I(\theta)\, v \cdot \text{Var}_\theta(\hat\varphi).
$$

Substituting $v = I(\theta)^{-1/2} u$ gives $\text{Var}_\theta(\hat\varphi) \geq (b^\top I(\theta)^{-1/2} u)^2$. Maximising over unit vectors $u$ by choosing $u \propto I(\theta)^{-1/2} b$ yields $\text{Var}_\theta(\hat\varphi) \geq b^\top I(\theta)^{-1} b$.

For an unbiased estimator $\hat\theta$ of $\theta \in \mathbb{R}^p$, apply this with $\varphi(\theta) = v^\top \theta$ for arbitrary $v$. Then $b = v$ and $\hat\varphi = v^\top \hat\theta$, giving $v^\top \text{Cov}_\theta(\hat\theta)\, v \geq v^\top I(\theta)^{-1} v$ for all $v$. This means $\text{Cov}_\theta(\hat\theta) - I(\theta)^{-1}$ is positive semi-definite. $\square$

### A5: Equality condition

Equality in the Cauchy–Schwarz step of A3 holds if and only if $\hat\varphi - \mathbb{E}_\theta[\hat\varphi]$ is proportional to $S(\theta)$. In the scalar case with i.i.d. data and an unbiased estimator $\hat\theta$, this requires

$$
\hat\theta = \theta + \frac{1}{n\, I_1(\theta)} \cdot S_n(\theta)
$$

for all $\theta$. This condition is quite restrictive and holds primarily for exponential family models with their natural sufficient statistics. When it does hold, the CRLB-achieving estimator is the unique UMVUE.

## Sources

The material in this post draws on the likelihood inference chapter of the Principles of Statistics course at the University of Cambridge.

- Rajen Shah, *Principles of Statistics*, University of Cambridge lecture notes, Section 1.2.
- George Casella and Roger L. Berger, *Statistical Inference*, 2nd edition, Section 7.3.

The full code for all figures is in the [Jupyter notebook on GitHub](https://github.com/alexwangjiaqi/alexwangjiaqi.github.io/blob/main/notebooks/fisher-crlb.ipynb).